In [2]:
import sys
from pathlib import Path

# Add the repo root (where utils.py lives) to the import path.
# Search upward from the current dir for the folder containing utils.py.
here = Path.cwd()
for p in [here, *here.parents]:
    if (p / "utils.py").exists():
        sys.path.insert(0, str(p))
        print("added to path:", p)
        break
else:
    print("utils.py not found searching up from", here)

import utils
print("utils loaded from:", utils.__file__)

added to path: C:\Users\conno\Documents\GitHub\ML_MetaLearning_SoftMatter
utils loaded from: C:\Users\conno\Documents\GitHub\ML_MetaLearning_SoftMatter\utils.py


In [5]:
import utils
basic_info = utils.load_basic_info()
file_names = utils.list_tasks()
clf, reg = utils.split_problem_lists(file_names, basic_info)
print(f"{len(file_names)} tasks | {len(clf)} classification | {len(reg)} regression")

out = utils.process_all_tasks(file_names=file_names, basic_info=basic_info)   # ~15-20 min

# ANCHOR 1
in_ct  = sum((out['in_splits'][t]['fold']=='train').sum()  for t in file_names)
out_ct = sum((out['out_splits'][t]['fold']=='train').sum() for t in file_names)
print(f"inner {in_ct/1e6:.3f}M + outer {out_ct/1e6:.3f}M = {(in_ct+out_ct)/1e6:.3f}M   (target 19.573M)")

# ANCHOR 2
shp = out['ia_os_merge']['ext_qual_binary'].shape
print(f"ext_qual_binary in_agg_out_splits_merge shape: {shp}   (target (135360, 48))")

31 tasks | 14 classification | 17 regression
--- ext_qual_binary
--- ext_qual_failure_mode
--- ext_qual_filament_outcome
--- ext_quant_avg_force
--- ext_quant_avg_force_under4
--- ext_quant_curves
--- ext_quant_max_force
--- ext_quant_max_force_under5
--- ext_quant_stability
--- ext_quant_std_force
--- microgel_formation
--- microgel_init_size
--- microgel_jamming_curves
--- microgel_jamming_curves_over08
--- microgel_shape
--- microgel_size_evolution
--- microgel_size_evolution_under300
--- rheo_osc_plateau_loss
--- rheo_osc_plateau_loss_under20
--- rheo_osc_plateau_storage
--- rheo_osc_plateau_storage_over1000
--- rheo_osc_strain_loss_curves
--- rheo_osc_strain_storage_curves
--- rheo_osc_stress_loss_curves
--- rheo_osc_stress_storage_curves
--- rheo_osc_yield_strain
--- rheo_osc_yield_strain_over10
--- rheo_osc_yield_stress
--- rheo_osc_yield_stress_over20
--- rheo_rot_shear_curves
--- rheo_rot_yield
inner 18.790M + outer 0.783M = 19.573M   (target 19.573M)
ext_qual_binary in_agg_ou

In [12]:
import pickle
from pathlib import Path

# REPO: reuse the path you already resolved in this notebook (where utils.py lives).
# If it's not still in scope, re-find it:
here = Path.cwd()
for p in [here, *here.parents]:
    if (p / "utils.py").exists():
        REPO = p; break

cache_dir = REPO / "cache"; cache_dir.mkdir(exist_ok=True)
cache_path = cache_dir / "processed_dicts.pkl"
with open(cache_path, "wb") as f:
    pickle.dump({"out": out, "file_names": file_names}, f, protocol=pickle.HIGHEST_PROTOCOL)
print("cached ->", cache_path, "|", round(cache_path.stat().st_size/1e6, 1), "MB")

cached -> C:\Users\conno\Documents\GitHub\ML_MetaLearning_SoftMatter\cache\processed_dicts.pkl | 16883.7 MB


In [13]:
with open(cache_path, "rb") as f:
    _c = pickle.load(f)
print("reload OK:", _c["out"]["ia_os_merge"]["ext_qual_binary"].shape,
      "| tasks:", len(_c["file_names"]))
# expect (135360, 48) | tasks: 31

reload OK: (135360, 48) | tasks: 31


In [14]:
# How big is each dict component? (rough: sum of memory across all tasks)
for key in out:
    if isinstance(out[key], dict):
        # dict of DataFrames
        try:
            mb = sum(v.memory_usage(deep=True).sum() for v in out[key].values())/1e6
            rows = sum(len(v) for v in out[key].values())
            print(f"{key:14s}: {mb:8.0f} MB | {rows:,} total rows")
        except AttributeError:
            print(f"{key:14s}: (non-frame dict)")

problem_type  : (non-frame dict)
scores        : (non-frame dict)
in_splits     :    20494 MB | 37,580,544 total rows
out_splits    :      748 MB | 1,565,856 total rows
in_agg        :     6185 MB | 7,829,280 total rows
out_agg       :      236 MB | 326,220 total rows
ia_oa_concat  :     6437 MB | 8,155,500 total rows
ia_os_merge   :     3210 MB | 3,914,640 total rows
ia_oa_merge   :     6107 MB | 3,914,640 total rows


In [15]:
import pickle
from pathlib import Path

cache_dir = REPO / "cache"; cache_dir.mkdir(exist_ok=True)

# small meta (file list + the two non-frame dicts) in one file
with open(cache_dir / "meta.pkl", "wb") as f:
    pickle.dump({"file_names": file_names,
                 "problem_type": out["problem_type"],
                 "scores": out["scores"]}, f, protocol=5)
print("meta.pkl written")

# one file per figure-usable frame-dict (reused across Paper 2 & 3)
for key in ("out_agg", "ia_os_merge", "ia_oa_merge"):
    path = cache_dir / f"{key}.pkl"
    with open(path, "wb") as f:
        pickle.dump(out[key], f, protocol=5)
    print(f"{key}.pkl:", round(path.stat().st_size/1e6, 1), "MB")

meta.pkl written
out_agg.pkl: 106.2 MB
ia_os_merge.pkl: 1467.4 MB
ia_oa_merge.pkl: 4356.5 MB


In [16]:
(REPO / "cache" / "processed_dicts.pkl").unlink(missing_ok=True)
print("old monolith removed")

old monolith removed


In [18]:
with open(cache_dir / "ia_oa_concat.pkl", "wb") as f:
    pickle.dump(out["ia_oa_concat"], f, protocol=5)
print("ia_oa_concat.pkl:", round((cache_dir/"ia_oa_concat.pkl").stat().st_size/1e6, 1), "MB")

ia_oa_concat.pkl: 2805.1 MB


In [19]:
for key in ("ia_oa_concat", "in_agg"):
    path = cache_dir / f"{key}.pkl"
    with open(path, "wb") as f:
        pickle.dump(out[key], f, protocol=5)
    print(f"{key}.pkl:", round(path.stat().st_size/1e6, 1), "MB")

ia_oa_concat.pkl: 2805.1 MB
in_agg.pkl: 2690.8 MB
